# Meridian / Model evaluation
Reproduce the SQL findings and the decisions for the July 2025 planning cycle. Each observation is an account-month.

In [1]:
from pathlib import Path
import sys, sqlite3
import pandas as pd
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
from analytics import ingest, run, sql
db = sqlite3.connect(':memory:')
with (root / 'data' / 'wholesale.csv').open() as file:
    ingest(db, file, 'Example Wholesale', 'Synthetic demo')
print(pd.read_sql_query(sql('quality'), db).to_string(index=False))

 raw_rows  duplicate_rows  missing_days  invalid_rows  clean_rows
     3271              31           156            16        3224


## SQL findings
How much was ordered, and what fraction of known next-month invoices was late? Monthly totals can change with account coverage.

In [2]:
print(pd.read_sql_query(sql('trends'),db).tail(6).to_string(index=False))

  month  units  accounts   balance  late_pct
2025-01  13505       179 147839.57      14.5
2025-02  14072       179 162751.56      17.9
2025-03  14965       179 163751.94      14.0
2025-04  15326       179 163536.57      21.2
2025-05  15719       179 177610.79      20.7
2025-06  15535       179 173999.43       NaN


## Model comparison
The training pipeline fits preprocessing on the earlier period. Permutation importance uses later labelled snapshots. The final model is then refit on all labelled history before scoring June accounts for July.

In [3]:
results = {key: run(db,key) for key in ('collections','demand')}
print(pd.DataFrame([{k:r[k] for k in ('title','metric','score','baseline','train','test','estimate')} for r in results.values()]).to_string(index=False))
print(pd.DataFrame(results['collections']['drivers']).to_string(index=False))

               title              metric  score  baseline  train  test  estimate
Collections priority   Average precision 0.4969    0.1765   2150   895      39.0
     Demand planning Mean absolute error 8.3388    8.1531   2150   895   15535.0
                      name   value
        Prior days overdue  0.2620
Three-period average units  0.1299
       Outstanding balance  0.0700
             Current units  0.0111
                    Region -0.0008


Average precision favours the collections model. Previous overdue days carry the most ranking information on this holdout. Run a reversible outreach pilot and measure precision among the accounts staff can contact.

For demand planning, repeating the current month’s units slightly beats ridge regression. Retain that planning baseline and evaluate seasonal features across rolling windows before deploying a replacement.

In [4]:
print(pd.DataFrame(results['collections']['predictions']).head(10).to_string(index=False))

account   region  prediction
  C0011   Tamale      0.9615
  C0098   Tamale      0.9419
  C0089   Kumasi      0.8953
  C0104   Tamale      0.7936
  C0014   Tamale      0.7868
  C0032   Kumasi      0.7779
  C0152   Kumasi      0.7549
  C0163   Tamale      0.7433
  C0042 Takoradi      0.7059
  C0114 Takoradi      0.6925


## Monitoring
Compare missingness, account mix and score distributions to the reference period at each ingestion. When labels mature, recompute average precision, calibration and precision at outreach capacity. For demand, track absolute error and signed bias by region against the persistence baseline. Record label delay and coverage before interpreting metric changes.